# Station 7: Das neuronale Netz – vorwärts rechnen, rückwärts lernen

Informatik 13 · Künstliche Intelligenz · edu-mrh.de · CC BY-NC-SA 4.0

Dieselben Programme wie auf der Website 7-neuronales-netz.html. Führe zuerst die Hilfszelle aus. Einrichtung von Python und Jupyter: https://edu-mrh.de/informatik/13/ki/python-einrichten.html

In [ ]:
# Hilfszelle: zuerst ausführen.
# Auf der Website zeichnet das Modul "ki" die Diagramme im Browser. Hier übernimmt das matplotlib.
# Datendateien (z. B. ring.csv) werden beim ersten Mal von edu-mrh.de geladen und neben dem Notebook gespeichert.
import os, urllib.request
import matplotlib.pyplot as plt

DATEN_URL = "https://edu-mrh.de/informatik/13/ki/data/"

def datei(name):
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATEN_URL + name, name)
    return name

class ki:
    @staticmethod
    def linie(x, reihen, titel="", xname="", yname=""):
        if not isinstance(reihen, dict):
            reihen = {yname or "Wert": list(reihen)}
        for name, werte in reihen.items():
            plt.plot(list(x), werte, marker="o" if len(werte) < 40 else None, label=name)
        if len(reihen) > 1:
            plt.legend()
        plt.title(titel); plt.xlabel(xname); plt.ylabel(yname); plt.grid(alpha=.3); plt.show()

    @staticmethod
    def streu(punkte, klassen=None, titel="", xname="x1", yname="x2"):
        klassen = klassen or [0] * len(punkte)
        plt.scatter([p[0] for p in punkte], [p[1] for p in punkte], c=klassen, cmap="coolwarm", s=18)
        plt.title(titel); plt.xlabel(xname); plt.ylabel(yname); plt.gca().set_aspect("equal"); plt.show()

    @staticmethod
    def region(f, punkte=(), klassen=None, schritte=40, titel="", xname="x1", yname="x2", bereich=(0, 1)):
        a, b = bereich
        gitter = [[f(a + (b - a) * (i + 0.5) / schritte, a + (b - a) * (j + 0.5) / schritte) for i in range(schritte)] for j in range(schritte)]
        plt.imshow(gitter, origin="lower", extent=(a, b, a, b), cmap="coolwarm", alpha=.35, vmin=0, vmax=1)
        if punkte:
            plt.scatter([p[0] for p in punkte], [p[1] for p in punkte], c=klassen, cmap="coolwarm", s=18)
        plt.title(titel); plt.xlabel(xname); plt.ylabel(yname); plt.show()

    @staticmethod
    def balken(namen, werte, titel="", xname="", yname=""):
        plt.bar([str(n) for n in namen], werte); plt.title(titel); plt.xlabel(xname); plt.ylabel(yname); plt.show()

    @staticmethod
    def tabelle(zeilen, kopf=None, titel=""):
        if titel:
            print(titel)
        if kopf:
            print(" | ".join(str(k) for k in kopf))
        for z in zeilen:
            print(" | ".join(str(round(v, 4)) if isinstance(v, float) else str(v) for v in z))

    @staticmethod
    def ziffern(bilder, namen=None, titel=""):
        fig, achsen = plt.subplots(1, len(bilder), figsize=(1.2 * len(bilder), 1.6))
        if len(bilder) == 1:
            achsen = [achsen]
        for k, (ax, bild) in enumerate(zip(achsen, bilder)):
            ax.imshow([bild[i * 8:(i + 1) * 8] for i in range(8)], cmap="gray_r", vmin=0, vmax=16)
            ax.axis("off")
            if namen:
                ax.set_title(str(namen[k]), fontsize=9)
        fig.suptitle(titel); plt.show()

for _name in ["pinguine.csv", "ziffern.csv", "ring.csv"]:
    try:
        datei(_name)
    except Exception as _fehler:
        print("Konnte", _name, "nicht laden:", _fehler)
print("Hilfszelle bereit.")


## Forward Propagation

**Idee:** Das XOR-Netz Schicht für Schicht ausrechnen, für alle vier Eingaben.
                            
1. `neuron(w, eingaben)` rechnet z = w0 + w1 · e1 + w2 · e2 und gibt σ(z) zurück.
1. ① Die verdeckte Schicht bekommt die Eingabe x, ② das Ausgabeneuron bekommt [h1, h2].
                            

**Achte auf:** Stimmen die Werte mit deiner Rechnung auf dem Blatt überein? Wie nah liegt y bei 0 bzw. 1? Ändere die Gewichte von w_y auf [-12, 8, 8]: Was passiert?

In [ ]:
import math

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

# Gewichte wie auf Blatt 7: [w0, w1, w2], w0 ist der Bias
w_h1 = [-2, 4, 4]
w_h2 = [6, -4, -4]
w_y = [-6, 4, 4]

def neuron(w, eingaben):
    z = w[0] + w[1] * eingaben[0] + w[2] * eingaben[1]
    return sigmoid(z)

def vorwaerts(x):
    h1 = neuron(w_h1, x)          # ① verdeckte Schicht
    h2 = neuron(w_h2, x)
    y = neuron(w_y, [h1, h2])     # ② Ausgabeschicht
    return h1, h2, y

for x in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    h1, h2, y = vorwaerts(x)
    print(x, "  h1 =", round(h1, 3), "  h2 =", round(h2, 3), "  y =", round(y, 3))

## Kostenfunktion

**Idee:** Für jedes Beispiel (y − Ziel)² berechnen, alles addieren, durch die Zahl der Beispiele teilen.
                            
1. Ergänze in `kosten` die Zeile mit dem Pfeil: statt `0` den quadrierten Fehler `(y - ziel) ** 2`.
1. Ausführen und mit deiner Rechnung auf dem Blatt vergleichen.
                            

**Achte auf:** Welches Beispiel trägt am meisten zu den Kosten bei? Ändere w_y auf [-12, 8, 8]: Werden die Kosten kleiner?

In [ ]:
import math

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

w_h1 = [-2, 4, 4]
w_h2 = [6, -4, -4]
w_y = [-6, 4, 4]

def neuron(w, eingaben):
    return sigmoid(w[0] + w[1] * eingaben[0] + w[2] * eingaben[1])

def vorwaerts(x):
    h1 = neuron(w_h1, x)
    h2 = neuron(w_h2, x)
    return neuron(w_y, [h1, h2])

daten = [((0, 0), 0), ((0, 1), 1), ((1, 0), 1), ((1, 1), 0)]   # (Eingabe, Ziel)

def kosten(daten):
    summe = 0
    for x, ziel in daten:
        y = vorwaerts(x)
        summe = summe + 0          # ← hier statt 0 den quadrierten Fehler (y - ziel) ** 2
    return summe / len(daten)

for x, ziel in daten:
    y = vorwaerts(x)
    print(x, "Ziel", ziel, "  y =", round(y, 3), "  (y - Ziel)² =", round((y - ziel) ** 2, 4))
print("Kosten K =", round(kosten(daten), 4))

## Lernen durch Wackeln (numerischer Gradient)

**Idee:** Das XOR-Netz soll seine 9 Gewichte selbst finden. Für jedes Gewicht messen wir die Steigung, indem wir es ein bisschen größer und kleiner machen (wie im Wackel-Experiment), und gehen dann bergab.
                            
1. `steigung(g, i)` wackelt Gewicht i um ± 0,001 und vergleicht die Kosten.
1. Jeder Schritt: alle 9 Steigungen messen, jedes Gewicht um −η · Steigung ändern.
1. Alle 100 Schritte werden die Kosten notiert und am Ende gezeichnet.
                            

**Achte auf:** Fällt die Kostenkurve? Wie oft wird `kosten` aufgerufen? Ändere `random.seed(3)` auf andere Zahlen: Klappt es immer?

In [ ]:
import math, random

random.seed(3)

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

daten = [((0, 0), 0), ((0, 1), 1), ((1, 0), 1), ((1, 1), 0)]

# alle 9 Gewichte in einer Liste: h1 = g[0..2], h2 = g[3..5], y = g[6..8]
g = [random.uniform(-1, 1) for i in range(9)]

def vorwaerts(g, x):
    h1 = sigmoid(g[0] + g[1] * x[0] + g[2] * x[1])
    h2 = sigmoid(g[3] + g[4] * x[0] + g[5] * x[1])
    return sigmoid(g[6] + g[7] * h1 + g[8] * h2)

aufrufe = 0
def kosten(g):
    global aufrufe
    aufrufe = aufrufe + 1
    return sum((vorwaerts(g, x) - ziel) ** 2 for x, ziel in daten) / len(daten)

def steigung(g, i, eps=0.001):
    plus = g[:]
    plus[i] = plus[i] + eps
    minus = g[:]
    minus[i] = minus[i] - eps
    return (kosten(plus) - kosten(minus)) / (2 * eps)

eta = 2.0
verlauf = []
for schritt in range(3000):
    if schritt % 100 == 0:
        verlauf.append(kosten(g))
    s = [steigung(g, i) for i in range(9)]
    g = [g[i] - eta * s[i] for i in range(9)]       # bergab

print("Kosten am Anfang:", round(verlauf[0], 4), "  am Ende:", round(kosten(g), 4))
for x, ziel in daten:
    print(x, "Ziel", ziel, "  y =", round(vorwaerts(g, x), 3))
print("Aufrufe von kosten:", aufrufe)
ki.linie(range(0, 3000, 100), verlauf, titel="Kosten beim Lernen", xname="Schritt", yname="Kosten")

## Backpropagation: Kern im Ring

**Idee:** Ein Netz mit 8 verdeckten Neuronen lernt per Backpropagation, den roten Kern vom blauen Ring zu trennen (das Level 5 aus dem Zaun-Baukasten, jetzt ohne Handarbeit).
                            
1. `vorwaerts` merkt sich die Werte aller Schichten (①).
1. `rueckwaerts`: Fehler am Ausgang (②), Schuld `delta` Schicht für Schicht zurück (③), Gewichte bergab (④).
1. 300 Epochen: jedes Beispiel einmal pro Epoche.
                            

**Achte auf:** Wie schnell sinken die Kosten im Vergleich zum Wackeln? Wie sieht die gelernte Region aus? Ändere die Zahl der verdeckten Neuronen (`Netz([2, 8, 1])`) auf 2 oder 3.

In [ ]:
import math, random, csv

random.seed(1)

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

class Netz:
    def __init__(self, groessen):
        # W[l][j] = [w0, w1, ...]: Bias und Gewichte von Neuron j in Schicht l
        self.W = []
        for l in range(1, len(groessen)):
            self.W.append([[random.uniform(-1, 1) for i in range(groessen[l - 1] + 1)] for j in range(groessen[l])])

    def vorwaerts(self, x):                                     # ①
        a = [list(x)]
        for schicht in self.W:
            a.append([sigmoid(w[0] + sum(wi * ai for wi, ai in zip(w[1:], a[-1]))) for w in schicht])
        return a

    def rueckwaerts(self, x, ziel, eta):
        a = self.vorwaerts(x)
        y = a[-1][0]
        delta = [[2 * (y - ziel) * y * (1 - y)]]                 # ② Fehler am Ausgang
        for l in range(len(self.W) - 1, 0, -1):                 # ③ Schuld rückwärts
            d = []
            for j in range(len(self.W[l - 1])):
                schuld = sum(self.W[l][k][j + 1] * delta[0][k] for k in range(len(self.W[l])))
                d.append(schuld * a[l][j] * (1 - a[l][j]))
            delta.insert(0, d)
        for l, schicht in enumerate(self.W):                    # ④ Gewichte bergab
            for j, w in enumerate(schicht):
                w[0] = w[0] - eta * delta[l][j]
                for i in range(len(w) - 1):
                    w[i + 1] = w[i + 1] - eta * delta[l][j] * a[l][i]
        return (y - ziel) ** 2

daten = [((float(z["x"]), float(z["y"])), int(z["klasse"])) for z in csv.DictReader(open(datei("ring.csv")))]
netz = Netz([2, 8, 1])
verlauf = []
for epoche in range(300):
    random.shuffle(daten)
    k = sum(netz.rueckwaerts(x, ziel, 1.0) for x, ziel in daten) / len(daten)
    if epoche % 10 == 0:
        verlauf.append(k)

richtig = sum(1 for x, ziel in daten if (netz.vorwaerts(x)[-1][0] >= 0.5) == (ziel == 1))
print("Kosten am Anfang:", round(verlauf[0], 4), "  am Ende:", round(k, 4))
print("richtig:", richtig, "von", len(daten))
ki.linie(range(0, 300, 10), verlauf, titel="Kosten je Epoche", xname="Epoche", yname="Kosten")
ki.region(lambda a, b: netz.vorwaerts((a, b))[-1][0], [x for x, _ in daten], [z for _, z in daten], titel="Gelernte Region (rot = Kern)")

## Zusatz: die numpy-Fassung
Dieselbe Rechnung wie in der Backpropagation-Zelle, aber mit Matrizen. Am eigenen Rechner ist numpy schnell; im Browser auf der Website gibt es nur die Standardbibliothek.

In [ ]:
# Zusatz: dasselbe Backpropagation-Netz mit numpy (so schreibt man es in echten Bibliotheken).
# Statt Schleifen über Neuronen rechnet numpy mit ganzen Matrizen auf einmal.
import numpy as np, csv

zeilen = list(csv.DictReader(open(datei("ring.csv"))))
X = np.array([[float(z["x"]), float(z["y"])] for z in zeilen])       # 160 x 2
T = np.array([[float(z["klasse"])] for z in zeilen])                  # 160 x 1

rng = np.random.default_rng(1)
W1, b1 = rng.uniform(-1, 1, (2, 8)), rng.uniform(-1, 1, 8)            # Eingabe -> verdeckt
W2, b2 = rng.uniform(-1, 1, (8, 1)), rng.uniform(-1, 1, 1)            # verdeckt -> Ausgabe
sig = lambda z: 1 / (1 + np.exp(-z))

# Hier wird über alle 160 Beispiele gemittelt und erst dann ein Schritt gemacht ("Batch").
# Darum braucht es eine größere Lernrate als auf der Website, wo nach jedem Beispiel gelernt wird.
eta = 10.0
verlauf = []
for epoche in range(3000):
    H = sig(X @ W1 + b1)                      # ① vorwärts, alle Beispiele auf einmal
    Y = sig(H @ W2 + b2)
    verlauf.append(float(np.mean((Y - T) ** 2)))
    dY = 2 * (Y - T) * Y * (1 - Y) / len(X)  # ② Fehler am Ausgang
    dH = (dY @ W2.T) * H * (1 - H)            # ③ Schuld rückwärts
    W2 -= eta * H.T @ dY; b2 -= eta * dY.sum(0)   # ④ Gewichte bergab
    W1 -= eta * X.T @ dH; b1 -= eta * dH.sum(0)

print("Kosten am Ende:", round(verlauf[-1], 4), " richtig:", int(np.sum((Y >= 0.5) == (T == 1))), "von", len(X))
plt.plot(verlauf); plt.xlabel("Schritt"); plt.ylabel("Kosten"); plt.show()
